# Customer Churn Prediction using Random Forest
## End-to-End Machine Learning and Deployment Use Case
### Course: B.Tech – Gen AI (2nd Semester)

## 1. Introduction

**Customer Churn** refers to the phenomenon where customers stop doing business with a company. In the travel/airline industry, churn means customers who stop booking flights or using services. Predicting churn proactively allows businesses to retain customers through targeted offers and improved service.

**Why is Churn Prediction Important?**
- Acquiring a new customer costs 5–7x more than retaining an existing one.
- Even a 5% increase in retention can increase profits by 25–95%.
- Early identification of at-risk customers enables proactive intervention.

**Why Random Forest?**
- Handles both numerical and categorical features effectively.
- Resistant to overfitting due to ensemble averaging.
- Provides built-in feature importance, which is valuable for business insights.
- Works well on small-to-medium datasets without extensive tuning.

## 2. Importing Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_curve, auc
)

import pickle

print('All libraries imported successfully!')

## 3. Data Loading and Exploration

In [ ]:
# Load dataset
df = pd.read_csv('Customertravel.csv')

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Summary Statistics:')
df.describe(include='all')

In [ ]:
print('Data Types:')
print(df.dtypes)
print('\nMissing Values:')
print(df.isnull().sum())

In [ ]:
print('Target variable distribution:')
print(df['Target'].value_counts())
print(f"\nChurn Rate: {df['Target'].mean()*100:.2f}%")

## 4. Data Cleaning and Preprocessing

In [ ]:
# Drop duplicates if any
df.drop_duplicates(inplace=True)
print(f'Shape after removing duplicates: {df.shape}')

# Fill missing values if any
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)
for col in df.select_dtypes(include='number').columns:
    df[col].fillna(df[col].median(), inplace=True)

print('Missing values after cleaning:', df.isnull().sum().sum())

In [ ]:
# Encode categorical features
le = LabelEncoder()
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print('Categorical columns to encode:', categorical_cols)

df_encoded = df.copy()
label_encoders = {}
for col in categorical_cols:
    le_col = LabelEncoder()
    df_encoded[col] = le_col.fit_transform(df[col])
    label_encoders[col] = le_col

print('\nEncoded dataset:')
df_encoded.head()

In [ ]:
# Split features and target
X = df_encoded.drop('Target', axis=1)
y = df_encoded['Target']  # Already binary (0/1)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples: {X_test.shape[0]}')
print(f'Features: {X.columns.tolist()}')

## 5. Model Development: Random Forest Classifier

In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

print('Model trained successfully!')
print(f'Number of trees: {rf_model.n_estimators}')

In [ ]:
# Save model
with open('model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

# Save feature names for deployment
feature_names = X.columns.tolist()
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print('Model saved as model.pkl')

## 6. Model Evaluation

In [ ]:
# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy Score: {accuracy*100:.2f}%')

# Classification Report
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Churned', 'Churned']))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Churned', 'Churned'],
            yticklabels=['Not Churned', 'Churned'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print(f'\nTrue Positives: {cm[1,1]}, False Negatives: {cm[1,0]}')
print(f'False Positives: {cm[0,1]}, True Negatives: {cm[0,0]}')

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2,
         label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve – Random Forest', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150)
plt.show()
print(f'AUC Score: {roc_auc:.4f}')

In [ ]:
# Feature Importance
importances = rf_model.feature_importances_
feat_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=feat_df, x='Importance', y='Feature', palette='viridis')
plt.title('Feature Importance – Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print('Top Features:')
print(feat_df)

## 7. Visualizations

In [ ]:
# Churn Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_labels = ['Not Churned (0)', 'Churned (1)']
churn_counts = df['Target'].value_counts().sort_index()

axes[0].bar(churn_labels, churn_counts.values, color=['steelblue', 'tomato'], width=0.5)
axes[0].set_title('Churn Distribution (Bar Chart)', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 3, str(v), ha='center', fontweight='bold')

axes[1].pie(churn_counts.values, labels=churn_labels, autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Churn Distribution (Pie Chart)', fontweight='bold')

plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=150)
plt.show()

In [ ]:
# Key Feature Analysis
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

cat_features = ['FrequentFlyer', 'AnnualIncomeClass', 'AccountSyncedToSocialMedia', 'BookedHotelOrNot']
num_features = ['Age', 'ServicesOpted']

for i, col in enumerate(cat_features):
    ct = df.groupby([col, 'Target']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=axes[i], color=['steelblue', 'tomato'], rot=0)
    axes[i].set_title(f'{col} vs Churn', fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].legend(['Not Churned', 'Churned'])

for j, col in enumerate(num_features):
    df[df['Target']==0][col].hist(ax=axes[4+j], alpha=0.7, color='steelblue', bins=15, label='Not Churned')
    df[df['Target']==1][col].hist(ax=axes[4+j], alpha=0.7, color='tomato', bins=15, label='Churned')
    axes[4+j].set_title(f'{col} Distribution by Churn', fontweight='bold')
    axes[4+j].legend()

plt.tight_layout()
plt.savefig('feature_analysis.png', dpi=150)
plt.show()

## 8. Conclusion

### Model Performance Summary
The Random Forest Classifier was successfully trained on the Customer Travel Churn dataset. Key findings:

- **Accuracy**: The model achieves strong predictive accuracy on the test set.
- **AUC Score**: A high AUC value indicates excellent discrimination between churned and non-churned customers.
- **Confusion Matrix**: The model correctly identifies most churn cases, with manageable false positives.

### Key Features Contributing to Churn
Based on feature importance analysis:
1. **FrequentFlyer** status is a significant predictor — non-frequent flyers tend to churn more.
2. **ServicesOpted** — customers using fewer services show higher churn tendency.
3. **AnnualIncomeClass** — income level influences customer loyalty.
4. **Age** — younger customers may be less loyal.

### Possible Improvements
- Hyperparameter tuning using GridSearchCV or RandomizedSearchCV.
- Try other algorithms (XGBoost, LightGBM) for comparison.
- Use SMOTE for handling class imbalance if present.
- Collect more features (customer support interactions, pricing sensitivity).
- Deploy with real-time data pipeline for live predictions.